# 03 — Data Cleaning
Dataset: IBM HR Analytics Employee Attrition  
Goal: identify and fix data quality issues, then save the clean dataset to data/processed/.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded: 1470 rows, 35 columns


## 3.1 Missing Values
Before any analysis, we check data integrity.
An unhandled missing value can distort means, correlations, and predictions.

In [2]:
# Count NaN values per column
missing = df.isnull().sum()

# Show only columns with at least 1 missing value
print("Columns with missing values:")
print(missing[missing > 0])

Columns with missing values:
Series([], dtype: int64)


## 3.2 Hidden Missing Values
No NaN found — but missing data can hide as strings like "N/A", "none", or empty spaces.
Let's check for suspicious values in categorical columns.

In [4]:
# Check for suspicious string values in object columns
suspicious = ['N/A', 'NA', 'none', 'None', 'unknown', 'Unknown', '', ' ']

for col in df.select_dtypes(include='str').columns:
    found = df[col].isin(suspicious).sum()
    if found > 0:
        print(f"{col}: {found} suspicious values")

print("Check complete.")

Check complete.


## Missing Values — Results
No missing values (NaN) were found in any column.
No hidden missing values were detected (e.g. "N/A", "none", empty strings).

The dataset is complete and ready for the next cleaning steps.

## 3.3 Useless Columns
Some columns may have only one unique value across all rows.
These carry no information and should be removed.

In [5]:
# Find columns with only one unique value
constant_cols = [col for col in df.columns if df[col].nunique() == 1]

print("Columns with a single unique value:")
print(constant_cols)

Columns with a single unique value:
['EmployeeCount', 'Over18', 'StandardHours']


In [6]:
# Drop columns with a single unique value — they carry no information
cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours']

df = df.drop(columns=cols_to_drop)

print(f"Columns removed: {cols_to_drop}")
print(f"Dataset shape after dropping: {df.shape}")

Columns removed: ['EmployeeCount', 'Over18', 'StandardHours']
Dataset shape after dropping: (1470, 32)


## Useless Columns — Results
Three columns were found with a single unique value across all 1,470 rows:

- `EmployeeCount`: always 1 — a row counter with no analytical value
- `Over18`: always "Y" — likely a legal hiring requirement, no variation
- `StandardHours`: always 80 — same contract for all employees

None of these columns can help explain why an employee leaves.
If a value is identical for everyone, it cannot discriminate between attrition and retention.

All three columns were removed. Dataset shape: 1,470 rows × 35 columns → 1,470 rows × 32 columns.

## 3.4 Data Types
Correct data types are essential for analysis.
A number stored as a string cannot be used in calculations or models.

In [7]:
# Check data types for all columns
print(df.dtypes)
print(f"\nNumeric columns: {df.select_dtypes(include='number').shape[1]}")
print(f"String columns: {df.select_dtypes(include='str').shape[1]}")

Age                         int64
Attrition                     str
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                        str
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                       str
JobSatisfaction             int64
MaritalStatus                 str
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
OverTime                      str
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StockOptionLevel            int64
TotalWorkingYears           int64
TrainingTimesLastYear       int64
WorkLifeBalance             int64
YearsAtCompany              int64
YearsInCurrent

## 3.5 Binary Columns Encoding
Columns like `Attrition`, `OverTime`, and `Gender` contain only two possible values.
Converting them to 0/1 makes them ready for correlation analysis and machine learning models.

In [8]:
# Check unique values in binary columns
binary_cols = ['Attrition', 'OverTime', 'Gender']

for col in binary_cols:
    print(f"{col}: {df[col].unique()}")

Attrition: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
OverTime: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Gender: <StringArray>
['Female', 'Male']
Length: 2, dtype: str


In [9]:
# Encode binary columns to 0/1
# Attrition: Yes=1, No=0
# OverTime: Yes=1, No=0
# Gender: Male=1, Female=0 (arbitrary convention, documented here)

df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# Verify the conversion
print(df[['Attrition', 'OverTime', 'Gender']].head(10))
print(f"\nData types after encoding:")
print(df[['Attrition', 'OverTime', 'Gender']].dtypes)

   Attrition  OverTime  Gender
0          1         1       0
1          0         0       1
2          1         1       1
3          0         1       0
4          0         0       1
5          0         0       1
6          0         1       0
7          0         0       1
8          0         0       1
9          0         0       1

Data types after encoding:
Attrition    int64
OverTime     int64
Gender       int64
dtype: object


## 3.4 Data Types — Results
The dataset contains 24 numeric columns and 8 string columns.

Three string columns were identified as binary variables requiring encoding:

- `Attrition`: "Yes" / "No" → 1 / 0
- `OverTime`: "Yes" / "No" → 1 / 0
- `Gender`: "Male" / "Female" → 1 / 0 (arbitrary convention, documented here)

Storing binary variables as strings makes them unusable for correlation analysis and machine learning models.
All three columns were converted to int64 using `.map()`.

## 3.6 Saving the Clean Dataset
The cleaned dataset is saved to data/processed/ for use in subsequent analysis steps.
Raw data is never modified — processed/ contains our certified, analysis-ready version.

In [10]:
# Save the clean dataset to data/processed/
df.to_csv('../data/processed/hr_attrition_clean.csv', index=False)

print("Clean dataset saved to data/processed/hr_attrition_clean.csv")
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")

Clean dataset saved to data/processed/hr_attrition_clean.csv
Final shape: 1470 rows, 32 columns


## Cleaning Summary
The following operations were performed on the raw dataset:

| Step | Action | Result |
|------|--------|--------|
| Missing values | No NaN found | No action needed |
| Hidden missing values | No suspicious strings found | No action needed |
| Useless columns | 3 constant columns removed | 35 → 32 columns |
| Binary encoding | 3 columns converted to int64 | `Attrition`, `OverTime`, `Gender` |

Clean dataset saved to `data/processed/hr_attrition_clean.csv`.
Ready for exploratory data analysis.